![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 14 -- Lab 1: Tokenization Hands-On

Before any AI model can read text, the text must be broken into smaller pieces called **tokens**. In this lab you will build tokenizers step by step -- starting with the simplest approach and working up to the same algorithm used by ChatGPT.

| Part | Goal |
|---|---|
| Part 1 | Split text into words using basic Python |
| Part 2 | Build a BPE tokenizer from scratch |
| Part 3 | Use professional tokenizer tools |
| Part 4 | Compare all approaches |

In [ ]:
!pip install tokenizers -q

from collections import Counter

---
## Part 1: Simple Word Tokenization

The simplest way to tokenize text is to split it into words. Let's start with a small collection of sentences.

In [ ]:
corpus = [
    "Artificial intelligence is changing the world.",
    "Deep learning models need large datasets.",
    "Natural language processing deals with text.",
    "Can machines understand language? That is the question.",
    "Tokenization is the first step in NLP.",
]

for i, sentence in enumerate(corpus):
    print(f"Sentence {i+1}: {sentence}")

### Task 1: Split by Spaces

Use Python's `.split()` method to break the first sentence into words. Print the list of tokens.

**Example:** `"hello world".split()` gives `["hello", "world"]`

In [ ]:
# Your code here

### Task 2: Spot the Problem

Look at your output from Task 1. You should see tokens like `"world."` with the period attached. That's a problem -- `"world"` and `"world."` are the same word but look different to the computer.

Let's fix this. The code below removes common punctuation before splitting. **Run it**, then compare the output with Task 1.

In [ ]:
def clean_and_split(text):
    """Lowercase the text, remove punctuation, and split into words."""
    text = text.lower()
    for punct in [".", ",", "!", "?", ";", ":", "'", '"']:
        text = text.replace(punct, "")
    return text.split()

sentence = corpus[0]
print(f"Original:  {sentence}")
print(f"Tokens:    {clean_and_split(sentence)}")

### Task 3: Tokenize the Whole Corpus

Use `clean_and_split()` on **every sentence** in the corpus. Collect all tokens into one big list called `all_tokens`.

Then print:
1. The total number of tokens
2. The number of **unique** tokens (vocabulary size) -- use `set(all_tokens)`

In [ ]:
# Your code here

### Task 4: Count Word Frequencies

Use `Counter(all_tokens)` to count how often each word appears. Print the **10 most common** words using `.most_common(10)`.

Which words appear most often? Are they meaningful or just common words like "the" and "is"?

In [ ]:
# Your code here

### Discussion: Why Word Tokenization Has Problems

Think about these questions (discuss with your partner):

1. If we had 1 million documents, how large would the vocabulary get?
2. What happens when the model sees a word it has never seen before (e.g., a typo like "langauge")?
3. Are "running" and "run" treated as the same token or different tokens?

These problems motivate **subword tokenization** -- instead of splitting into whole words, we split into smaller pieces.

---
## Part 2: Build a BPE Tokenizer From Scratch

**BPE** (Byte Pair Encoding) is the tokenization algorithm used by GPT and many modern AI models.

**How it works:**
1. Start by splitting every word into individual **characters**
2. Count which **pair of adjacent characters** appears most often
3. **Merge** that pair into a single new token
4. Repeat steps 2-3 many times

After training, common words stay whole (like "the") while rare words get broken into known pieces (like "un" + "like" + "ly").

### Step 1: Start With Characters (GIVEN)

We begin with a small corpus. Every word is split into characters with a special `</w>` marker at the end.

In [ ]:
corpus = [
    "low low low low low",
    "lower lower wider wider",
    "newest newest newest newest newest newest",
    "widest widest widest",
]


def get_word_freqs(corpus):
    """Split each word into characters and count how often each word appears."""
    word_freqs = {}
    for sentence in corpus:
        for word in sentence.split():
            chars = list(word) + ["</w>"]
            word_key = tuple(chars)
            word_freqs[word_key] = word_freqs.get(word_key, 0) + 1
    return word_freqs


word_freqs = get_word_freqs(corpus)

print("Initial vocabulary (each word split into characters):")
print()
for word, freq in word_freqs.items():
    print(f"  {' '.join(word)}  (appears {freq} times)")

### Task 5: Count Pairs

Write a function `get_pair_counts(word_freqs)` that counts how often each pair of **neighboring tokens** appears.

For example, in `('l', 'o', 'w', '</w>')` which appears 5 times, the pairs are:
- `('l', 'o')` -- count 5
- `('o', 'w')` -- count 5
- `('w', '</w>')` -- count 5

**Hint:** Loop through each word. For each position `i`, the pair is `(word[i], word[i+1])`. Multiply by the word's frequency.

```python
def get_pair_counts(word_freqs):
    counts = Counter()
    for word, freq in word_freqs.items():
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            counts[pair] += freq
    return counts
```

Call the function and print the 5 most common pairs.

In [ ]:
# Your code here

### Step 2: Merge Function (GIVEN)

This function takes the most frequent pair and merges it everywhere in the vocabulary.

In [ ]:
def merge_pair(pair, word_freqs):
    """Merge all occurrences of a pair into a single token."""
    new_freqs = {}
    for word, freq in word_freqs.items():
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
                new_word.append(word[i] + word[i + 1])
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_freqs[tuple(new_word)] = freq
    return new_freqs

### Task 6: Run the BPE Training Loop

Now put it all together. For 10 iterations:
1. Get pair counts
2. Find the most common pair
3. Print what is being merged
4. Merge it

The starter code is below -- fill in the missing lines.

In [ ]:
num_merges = 10
merges = []

for step in range(num_merges):
    # 1. Get pair counts
    # Your code here

    # 2. Find the most common pair
    # Your code here (hint: use .most_common(1))

    # 3. Print the merge
    print(f"Merge {step + 1}: {best_pair[0]} + {best_pair[1]} -> {best_pair[0] + best_pair[1]}")

    # 4. Merge the pair and save
    merges.append(best_pair)
    word_freqs = merge_pair(best_pair, word_freqs)

print("\nFinal vocabulary after 10 merges:")
for word, freq in word_freqs.items():
    print(f"  {' '.join(word)}  ({freq})")

### Task 7: Tokenize a New Word

Now let's use our trained BPE to tokenize a word it has never seen. The function below applies the merges we learned.

**Run it**, then try tokenizing `"lowest"`, `"newest"`, and `"wider"`. Which parts does BPE recognize?

In [ ]:
def tokenize_bpe(word, merges):
    """Tokenize a word using learned BPE merges."""
    tokens = list(word) + ["</w>"]
    for pair in merges:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                new_tokens.append(tokens[i] + tokens[i + 1])
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens

# Try it:
for test_word in ["lowest", "newest", "wider"]:
    tokens = tokenize_bpe(test_word, merges)
    print(f"  {test_word} -> {tokens}")

### Task 8: Try Your Own Words

Use `tokenize_bpe()` to tokenize these words: `"widest"`, `"lower"`, `"high"`, `"highest"`.

Which words get split into pieces the BPE recognizes? Which ones stay mostly as individual characters? Why?

In [ ]:
# Your code here

---
## Part 3: Professional Tokenizers

In practice, we use libraries that implement BPE and other algorithms efficiently. The `tokenizers` library from HuggingFace can train a tokenizer in seconds.

### Training a BPE Tokenizer (GIVEN)

Run the cell below to train a BPE tokenizer on a small corpus.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

texts = [
    "The quick brown fox jumps over the lazy dog",
    "Natural language processing is a subfield of artificial intelligence",
    "Tokenization is the first step in any NLP pipeline",
    "Word embeddings capture semantic meaning of words",
    "Machine learning models need numerical input not raw text",
    "The transformer architecture revolutionized NLP in recent years",
    "Attention mechanisms allow models to focus on relevant parts",
    "Pre-trained language models like BERT use subword tokenization",
] * 50

# Train BPE tokenizer
bpe_tokenizer = Tokenizer(models.BPE())
bpe_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
bpe_trainer = trainers.BpeTrainer(vocab_size=200, special_tokens=["[UNK]", "[PAD]"])
bpe_tokenizer.train_from_iterator(texts, bpe_trainer)

print(f"BPE vocabulary size: {bpe_tokenizer.get_vocab_size()}")

# Train WordPiece tokenizer
wp_tokenizer = Tokenizer(models.WordPiece(unk_token="[UNK]"))
wp_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
wp_trainer = trainers.WordPieceTrainer(vocab_size=200, special_tokens=["[UNK]", "[PAD]"])
wp_tokenizer.train_from_iterator(texts, wp_trainer)

print(f"WordPiece vocabulary size: {wp_tokenizer.get_vocab_size()}")
print("\nBoth tokenizers trained!")

### Task 9: Compare Tokenizers

The function below encodes a sentence with all three methods. Run it on the test sentences and observe the differences.

Then answer: which tokenizer produces the fewest tokens? Which handles the long unseen word best?

In [ ]:
def compare_tokenizers(sentence):
    """Show how each tokenizer breaks a sentence into tokens."""
    # Whitespace
    ws_tokens = sentence.split()
    # BPE
    bpe_out = bpe_tokenizer.encode(sentence)
    # WordPiece
    wp_out = wp_tokenizer.encode(sentence)

    print(f"Sentence: \"{sentence}\"")
    print(f"  Whitespace ({len(ws_tokens)} tokens): {ws_tokens}")
    print(f"  BPE        ({len(bpe_out.tokens)} tokens): {bpe_out.tokens}")
    print(f"  WordPiece  ({len(wp_out.tokens)} tokens): {wp_out.tokens}")
    print()

compare_tokenizers("Machine learning models process language efficiently")
compare_tokenizers("antidisestablishmentarianism is a very long word")
compare_tokenizers("I love natural language processing")

### Task 10: Try Your Own Sentences

Use `compare_tokenizers()` to test at least 3 sentences of your own. Try:
- A sentence with a made-up word (e.g., "The flurpinator beeped loudly")
- A sentence in a different style (e.g., very formal or very casual)
- A sentence with a misspelled word

In [ ]:
# Your code here

---
## Part 4: Discussion

Answer these questions (discuss with your partner or write your answers below):

1. Why does BPE produce fewer tokens than word-level for common sentences?

2. What happens when BPE encounters a word it has never seen? (Hint: it breaks it into smaller known pieces.)

3. Why do modern AI models (GPT, BERT) use subword tokenization instead of whole-word tokenization?

*Write your answers here.*

---
## Wrap-Up

**Key takeaways:**

- Tokenization splits text into pieces that a model can process
- Word-level tokenization is simple but creates huge vocabularies and can't handle new words
- BPE starts from characters and learns common subwords -- this is what GPT uses
- WordPiece is a similar algorithm used by BERT
- Subword tokenizers handle any word, even ones they have never seen, by breaking them into known pieces

**Next:** In Lab 2, you will load pre-trained word embeddings and explore the vector space.